# Empathetic Conversational Recommender Systems — Experiment Notebook

Notebook **simple et modulaire** qui re-execute l'experience de l'article :

> Xiaoyu Zhang, Ruobing Xie, Yougang Lyu, Xin Xin, Pengjie Ren, Mingfei Liang, Bo Zhang, Zhanhui Kang, Maarten de Rijke, Zhaochun Ren.  
> **Towards Empathetic Conversational Recommender Systems**. *RecSys '24*, Bari, Italy — [doi:10.1145/3640457.3688133](https://doi.org/10.1145/3640457.3688133).

Le code de reference est celui du depot officiel [zxd-octopus/ECR](https://github.com/zxd-octopus/ECR). Ce notebook se structure comme l'exemple [WebSemantic-Projet-n-1/projet_session/tag_reco_experiment.ipynb](https://github.com/WebSemantic-Projet-n-1/projet_session/blob/main/tag_reco_experiment.ipynb) :

- **Une fonction par cellule** (pas de logique inline).
- **Une seule cellule d'import de donnees**.
- **Tous les graphiques** sont factorises dans `src/viz/plots.py`.
- **Derniere cellule** : compilation des resultats et visualisations comparatives.

### Plan du notebook

1. Parametrages globaux (hyperparametres de l'article, flags d'execution).
2. Preparation de l'environnement (clone du depot ECR + archives `emo_data` / `ckpt`).
3. Pretraitement du dataset **ReDial** (Section 4.1).
4. Sous-tache **Emotional Semantic Fusion** — `train_pre.py` (Section 4.1).
5. Sous-tache **Emotion-aware Item Recommendation** — `train_rec.py` (Section 4.2).
6. Sous-tache **Emotion-aligned Response Generation** — `train_emp.py` + `infer_emp.py` (Section 4.3).
7. Chargement des metriques objectives / subjectives (Tables 1-3).
8. Pipeline principal (import des donnees en **une seule cellule**).
9. Compilation finale + visualisations comparatives.

> **Note pratique.** L'entrainement complet de ECR demande un GPU 24 GB et plusieurs heures. Un drapeau `DRY_RUN` permet de ne lancer que la preparation legere du pipeline tout en reproduisant fidelement les tableaux de l'article (valeurs de repli).

In [ ]:
# Cellule 1 - Detection materielle (aligne avec la stack PyTorch de l'article)
import os

FORCE_CPU = False

if FORCE_CPU:
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"

try:
    import torch
    _torch_ok = True
except ImportError:
    _torch_ok = False

if _torch_ok:
    has_cuda = (not FORCE_CPU) and torch.cuda.is_available()
    device = "cuda" if has_cuda else "cpu"
    print(f"torch {torch.__version__} - device: {device}")
    if has_cuda:
        print(f"CUDA device: {torch.cuda.get_device_name(0)}")
        print(f"Memoire GPU: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    a = torch.randn(1024, 1024, device=device)
    _ = (a @ a).sum().item()
    print(f"Sanity matmul OK sur {device}.")
else:
    print("torch n'est pas installe - installez-le avec `pip install -r requirements.txt` si vous souhaitez re-entrainer les modeles.")

In [ ]:
# Cellule 2 - Imports (stack notebook + visualisations)
import sys
import shutil
import subprocess
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.viz.plots import (
    plot_ablation_study,
    plot_emotion_label_distribution,
    plot_feedback_distribution,
    plot_feedback_weights,
    plot_hyperparam_sweep,
    plot_llm_vs_human_correlation,
    plot_model_rankings,
    plot_objective_metrics,
    plot_review_coverage,
    plot_subjective_metrics,
    plot_subjective_radar,
    plot_training_loss,
)

print(f"Project root: {ROOT}")

## 1. Parametrages globaux

Les hyperparametres reprennent la **Section 5.4** de l'article :

- taille d'embedding des labels d'emotion : `48` ;
- seuil de retention `beta = 0.1`, nombre d'emotions partagees `n_f = 3` ;
- triplets / entites injectes dans le prompt : `p_nt = 2`, `p_ne = 4` ;
- **feedback-aware reweighting** (Eq. 7) : `like=2.0`, `dislike=1.0`, `not say=0.5` ;
- batch size `128` pour la recommandation, `16` pour la generation ;
- `lr = 1e-4` (DialoGPT) / `5e-5` (Llama 2-Chat + LoRA).

### Flags d'execution

| Flag | Defaut | Effet |
|---|---|---|
| `dry_run` | `True` | Court-circuite clone / archives / trainings / inference -> juste EDA + tables article. |
| `use_pretrained_ckpt` | `False` | Copie `data/ckpt/{pre,rec,emp}` dans `src_emo/data/saved/` et saute les 3 `train_*.py`. `infer_emp.py` est toujours lance. |
| `batch_scale` | `1.0` | Multiplicateur du `per_device_train_batch_size`. Le `gradient_accumulation_steps` est divise d'autant, donc l'**effective batch reste identique** a l'article. Valeur > 1.0 utile si GPU > 24 GB (ex. RTX 5090 32 GB -> `2.0` a tester). |
| `mixed_precision` | `"no"` | `"bf16"` recommande sur Blackwell / Ampere / Ada (≈ x1.5-2 wall-time, perte negligeable). `"fp16"` sinon. |

Les reglages **par defaut reproduisent fidelement l'article**. Les deux derniers flags sont des leviers hardware sans impact sur le LR ni la semantique du training.

In [ ]:
# Cellule 3 - Parametres globaux du pipeline ECR
def get_config():
    return {
        # Chemins principaux
        "root": ROOT,
        "ecr_repo_url": "https://github.com/zxd-octopus/ECR.git",
        "ecr_dir": ROOT / "ECR",
        "src_emo_dir": ROOT / "ECR" / "src_emo",
        "emo_data_archive": ROOT / "data" / "emo_data.zip",
        "emo_data_dir": ROOT / "data" / "emo_data",
        "ckpt_archive": ROOT / "data" / "ckpt.zip",
        "ckpt_dir": ROOT / "data" / "ckpt",
        "results_dir": ROOT / "results",
        "logs_dir": ROOT / "logs",
        "generations_dir": ROOT / "results" / "generations",
        # Fichiers de metriques (optionnels : repli sur valeurs de l'article sinon)
        "objective_file": "objective_metrics.csv",
        "subjective_llm_file": "subjective_metrics_llm.csv",
        "subjective_human_file": "subjective_metrics_human.csv",
        "ablation_file": "ablation_metrics.csv",
        # Archives externes (Google Drive - publiees par les auteurs ECR)
        # ~111 MB / ~679 MB : gdown gere automatiquement l'interstitiel > 100 MB.
        "emo_data_gdrive_id": "1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_",
        "ckpt_gdrive_id":     "1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19",
        # Hyperparametres article (Section 5.4)
        "emotion_embed_dim": 48,
        "beta": 0.1,
        "n_f": 3,
        "p_nt": 2,
        "p_ne": 4,
        "feedback_weights": {"like": 2.0, "dislike": 1.0, "not say": 0.5},
        "lr_dialogpt": 1e-4,
        "lr_llama": 5e-5,
        "batch_rec": 128,
        "batch_gen": 16,
        "seed": 42,
        # Seed fixe pour TOUS les 5 runs de Table 1/3 (sinon lignes non comparables).
        # Upstream ECR utilise seed=8 dans `train_rec.py` ; on garde cette valeur.
        "seed_rec_runs": 8,
        # Flags d'execution
        "dry_run": False,              # True  = aucun training/inference, juste EDA + tables article
        "use_pretrained_ckpt": False,  # True  = court-circuite les 3 trainings avec les poids data/ckpt/*
                                       #         (utile pour ne relancer que infer_emp.py et gagner >20h GPU)
        "skip_clone": False,
        "skip_download": False,
        # --- Flags Phase 2/3/4 : chaque etape peut etre desactivee independamment ---
        "run_rec_sweep":       True,   # Table 1 (ligne ECR + UniCRS) + Table 3 complete (5 variantes)
        "run_infer_emp":       True,   # ECR[DialoGPT] generation pour Table 2
        "run_llama_zero_shot": True,   # Llama 2-7B-Chat zero-shot pour Table 2
        "run_llama_lora":      True,   # ECR[Llama 2-Chat] LoRA fine-tune + generation (Table 2)
        "run_llm_scorer":      True,   # Qwen2.5-32B-AWQ scoring Appendix E (Table 2 LLM)
        "run_kappa":           True,   # Cohen kappa scorer local vs humains article
        # Optimisations hardware (profite d'un GPU > 24 GB comme la RTX 5090 32 GB).
        # L'article a ete entraine sur 24 GB : valeurs par defaut => fidelite article.
        # batch_scale > 1.0 : augmente `per_device_train_batch_size` et divise
        # `gradient_accumulation_steps` par le meme facteur. L'effective batch
        # reste identique a celui de l'article -> pas besoin de retuner le LR.
        # mixed_precision: "no" (article), "bf16" (recommande Blackwell), "fp16".
        "batch_scale": 2.0,
        "mixed_precision": "bf16",
        # Optimisations accelerations (Patches 22/23 + Flash-Attn 2 + vLLM Llama + scorer concurrent)
        "dataloader_num_workers": 8,      # Patch 22 : DataLoader parallele (9950X3D 16c)
        "use_torch_compile": True,        # Patch 23 : torch.compile(prompt_encoder)
        "use_flash_attn_2": True,         # Llama 2 inference/train : flash_attention_2 kwarg (fallback sdpa)
        "scorer_concurrency": 16,         # Qwen scorer : N requetes en vol via ThreadPoolExecutor
        "llama_vllm_autostart": False,    # True = auto-demarrer vllm serve Llama (besoin ~15 GB VRAM libre)
        # Modeles de base requis par les scripts ECR (Section 4) :
        # `AutoTokenizer.from_pretrained("save/dialogpt/")` etc. -> repertoires locaux.
        "base_models": {
            "microsoft/DialoGPT-small": "save/dialogpt",
            "roberta-base":             "save/roberta",
        },
        # --- Llama 2-7B-Chat (ECR[Llama 2-Chat] + zero-shot baseline) --------
        "llama_base_model":   "meta-llama/Llama-2-7b-chat-hf",
        "llama_local_dir":    ROOT / ".cache" / "huggingface" / "hub" / "llama2-7b-chat",
        "llama_lora_dir":     ROOT / "data" / "saved" / "ecr_llama_lora",
        "lora_r":             16,
        "lora_alpha":         32,
        "lora_dropout":       0.05,
        "lora_target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
        "lora_epochs":        15,
        "lora_per_device_batch": 4,
        "lora_grad_accum":    4,   # effective batch = 16 (fidele Section 5.4)
        "lora_use_4bit":      False,  # True si OOM (RTX 5090 32 GB : bf16 OK)
        # --- Scorer local (remplace GPT-4-turbo Section 5.6 / Appendix E) ----
        "scorer_model":       "Qwen/Qwen2.5-32B-Instruct-AWQ",
        "scorer_port":        8000,
        "scorer_sample_size": 200,    # taille panel humain article
        "scorer_max_model_len": 4096,
        # --- vLLM serveur Llama (sert les 2 generations rapidement) ----------
        "llama_serve_port":   8001,
        "llama_backend":      "hf",   # "hf" ou "vllm" (si vllm leve un 2e serveur)
    }


def accelerate_launch_cmd(cfg, script):
    """Construit `accelerate launch [--mixed_precision X] script` selon la config."""
    cmd = ["accelerate", "launch"]
    mp = cfg.get("mixed_precision", "no")
    if mp and mp != "no":
        cmd += ["--mixed_precision", mp]
    cmd.append(script)
    return cmd


def scaled_batch(cfg, per_device_batch, grad_accum):
    """Scale per_device_batch par cfg['batch_scale'] en reduisant grad_accum.

    Invariant strict : `new_batch * new_grad_accum == per_device_batch * grad_accum`.
    Si aucune solution entiere ne preserve l'effective batch (ex. grad_accum=1),
    on RENVOIE la paire d'origine pour rester fidele a l'article (sinon le LR
    calibre sur l'eff batch devient invalide).

    Patch post-conversation : l'ancienne version retournait `(40, 1)` pour
    (20, 1, scale=2.0), ce qui doublait l'eff batch sans prevenir. Desormais
    on detecte ce cas et on ne scale pas.
    """
    scale = max(1.0, float(cfg.get("batch_scale", 1.0)))
    if scale == 1.0:
        return per_device_batch, grad_accum
    effective = per_device_batch * grad_accum
    new_batch = max(1, int(round(per_device_batch * scale)))
    if new_batch > effective:
        print(f"[scaled_batch] scale={scale} trop agressif pour "
              f"(batch={per_device_batch}, accum={grad_accum}) -> pas de scaling")
        return per_device_batch, grad_accum
    new_grad_accum = effective // new_batch
    if new_batch * new_grad_accum != effective:
        print(f"[scaled_batch] effective batch non preservable avec scale={scale} "
              f"pour (batch={per_device_batch}, accum={grad_accum}) -> pas de scaling")
        return per_device_batch, grad_accum
    assert new_batch * new_grad_accum == effective, "scaled_batch invariant broken"
    return new_batch, new_grad_accum


cfg = get_config()
cfg

## 2. Preparation de l'environnement

Nous clonons le depot [zxd-octopus/ECR](https://github.com/zxd-octopus/ECR) puis telechargeons les deux archives publiees par les auteurs sur Google Drive :

- [`emo_data.zip`](https://drive.google.com/file/d/1fb9kDo8uSRLlwc5c4nUw8DZHR5XOY_l_/view) (~111 MB) — donnees emotionnelles pretraitees (annotations GPT-3.5 + reviews IMDb filtrees — Section 4.1). Contient `llama_train.json`, `llama_test.json`, les entites DBpedia et les reviews retenues.
- [`ckpt.zip`](https://drive.google.com/file/d/1uBtcqbQByVrrJ1hEwk2dvsAOxuvEgE19/view) (~679 MB) — poids entraines (GPT-2 classifieur d'emotions + ECR[DialoGPT] + ECR[Llama 2-Chat]).

Le telechargement utilise [`gdown`](https://github.com/wkentaro/gdown) qui gere automatiquement l'interstitiel Google Drive "virus scan warning" servi pour tout fichier > 100 MB. En cas d'echec reseau, deposer manuellement les deux `.zip` dans `data/` puis relancer la cellule.

In [ ]:
# Cellule 4 - Clone du depot officiel ECR
def clone_ecr_repo(cfg):
    """Clone le depot zxd-octopus/ECR a la racine du projet (idempotent)."""
    if cfg["skip_clone"] or cfg["ecr_dir"].exists():
        print(f"ECR repo deja present: {cfg['ecr_dir']}")
        return cfg["ecr_dir"]
    print(f"Clonage de {cfg['ecr_repo_url']} -> {cfg['ecr_dir']}")
    result = subprocess.run(
        ["git", "clone", "--depth", "1", cfg["ecr_repo_url"], str(cfg["ecr_dir"])],
        capture_output=True,
        text=True,
        check=False,
    )
    print(result.stdout)
    if result.returncode != 0:
        print("[STDERR]", result.stderr)
    return cfg["ecr_dir"]

### 2ter. Patches de compatibilite stack moderne

Le code ECR a ete ecrit pour `transformers 4.15` / `pyg 2.0.1` (fin 2023). Plusieurs APIs ont bouge depuis :

| Symbole | Statut | Correctif |
|---|---|---|
| `transformers.AdamW` | Retire en 4.30+ | -> `torch.optim.AdamW` (impl fusee CUDA plus rapide) |
| `transformers.modeling_utils.find_pruneable_heads_and_indices` | Deplace 4.17 | -> `transformers.pytorch_utils` |
| `transformers.utils.model_parallel_utils` | Retire 4.40+ | Stubs no-op (GPT-2 `parallelize()` non utilise par ECR) |
| `pyg.typing.SparseTensor` | Requiert `torch-sparse` | Format standard Tensor[2,E] + edge_type separe (compat RGCNConv) |

`patch_ecr_compat(cfg)` applique 5 patches idempotents sur les fichiers clones dans `ECR/src_emo/`. Aucun PR upstream ni download externe requis.

In [ ]:
# Cellule 4b - Compatibilite stack moderne (transformers >= 4.40 / pyg >= 2.5 / accelerate >= 0.20 / torch >= 2)
# ECR a ete ecrit pour transformers 4.15 / pyg 2.0.1 / accelerate 0.8 / torch 1.8.
# On applique des patches idempotents aux fichiers clones. Aucune PR upstream requise ;
# un `cd ECR && git checkout -- src_emo/` restaure l'original et patch_ecr_compat
# se re-applique au run suivant.
import re as _re


def _patch_file(path, replacements, description):
    if not path.exists():
        print(f"[patch] SKIP {description}: {path} absent")
        return
    text = path.read_text()
    original = text
    for pat, repl in replacements:
        text = _re.sub(pat, repl, text, flags=_re.MULTILINE)
    if text == original:
        print(f"[patch] {description}: deja applique (idempotent)")
        return
    path.write_text(text)
    print(f"[patch] {description}: OK")


def patch_ecr_compat(cfg):
    """Rend le code ECR compatible avec transformers >= 4.40 et pyg >= 2.5.

    Patches appliques (tous idempotents) :
      1. `transformers.AdamW` retire -> `torch.optim.AdamW` (train_{pre,rec,emp}.py)
      2. `modeling_utils.find_pruneable_heads_and_indices` -> `pytorch_utils`
      3. `model_parallel_utils` retire en 4.40 -> stubs no-op (jamais appele)
      4. `SparseTensor` (pyg >= 2.5 exige torch-sparse) -> Tensor [2,E] + edge_type
      5. RGCNConv call dans model_prompt.py -> forme standard (x, edge_index, edge_type)
      6. `Accelerator(..., fp16=...)` retire en accelerate 0.12 -> drop kwarg (mixed_precision via `accelerate launch`)
      7. `torch.set_deterministic` retire en torch 2.0 -> `torch.use_deterministic_algorithms(True, warn_only=True)`
         + CUDA_LAUNCH_BLOCKING / CUBLAS_WORKSPACE_CONFIG commentes (serialisation kernels -> 5-10x plus lent)
      8. `accelerator.use_fp16` retire en accelerate 0.20 -> getattr fallback (bf16 via launch)
      9. `.cuda()` hardcode dans dataset_dbpedia.py -> `.to(device)` pour tolerer CPU transitoire
     10. `print("here")` debug de l'auteur supprime (5 locations : train_pre/dataset_emp/imdb_filter/co_appear)
     11. `evaluate_rec.py` cree `save/redial_rec/` avant ecriture de `rec.json`
     12. `transformers.file_utils.ModelOutput` deprecated 4.25 (retire 5.0) -> `transformers.utils.ModelOutput`
     13. `PromptGPT2forCRS` herite aussi de `GenerationMixin` (transformers >= 4.50)
     14. `train_rec.py` ne force plus `CUDA_VISIBLE_DEVICES=3`
     15. `log/` est cree avant `logger.add(...)` dans train_{pre,rec,emp}.py + infer_emp.py
     16. chemins de sortie stabilises (`pre`, `rec`, `emp`) sans suffixe timestamp
     17. `train_pre.py` sauvegarde par defaut dans `data/saved/pre`
     18. `train_emp.py` sauvegarde par defaut dans `data/saved/emp`
     19. `infer_emp.py` charge localement (`local_files_only=True`) si le path modele existe
     20. suppression de `retain_graph=True` dans train_pre/train_rec
    """
    src_emo = cfg["src_emo_dir"]
    if not src_emo.exists():
        print(f"src_emo absent ({src_emo}) -> patches ignores.")
        return

    # Patch 1 : AdamW dans les 3 scripts de training
    adamw_patch = [(
        r"from transformers import AdamW, get_linear_schedule_with_warmup, AutoTokenizer, AutoModel",
        "from transformers import get_linear_schedule_with_warmup, AutoTokenizer, AutoModel\nfrom torch.optim import AdamW",
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py"):
        _patch_file(src_emo / name, adamw_patch, description=f"AdamW -> torch.optim ({name})")

    # Patches 2 + 3 : model_gpt2.py
    _patch_file(src_emo / "model_gpt2.py", [
        (
            r"from transformers\.modeling_utils import find_pruneable_heads_and_indices, prune_conv1d_layer",
            "from transformers.pytorch_utils import find_pruneable_heads_and_indices, prune_conv1d_layer",
        ),
        (
            r"^from transformers\.utils\.model_parallel_utils import assert_device_map, get_device_map\s*$",
            (
                "try:\n"
                "    from transformers.utils.model_parallel_utils import assert_device_map, get_device_map\n"
                "except ImportError:  # transformers >= 4.40 a retire le parallelisme custom GPT-2\n"
                "    def assert_device_map(*a, **kw): return None\n"
                "    def get_device_map(*a, **kw): return {}\n"
            ),
        ),
    ], description="model_gpt2.py imports")

    # Patch 4 : SparseTensor dans dataset_dbpedia.py
    _patch_file(src_emo / "dataset_dbpedia.py", [(
        r"^(\s*)self\.edge_index = SparseTensor\(row=self\.edge_index\[0\], col=self\.edge_index\[1\], value=self\.edge_type\)\s*$",
        r"\1# PATCH: SparseTensor supprime (PyG >= 2.5 requiert torch-sparse).\n"
        r"\1# edge_index reste Tensor [2,E], edge_type reste Tensor [E] -> format PyG standard.\n"
        r"\1# (voir model_prompt.py pour l'appel RGCNConv correspondant)",
    )], description="dataset_dbpedia.py SparseTensor")

    # Patch 5 : RGCNConv call dans model_prompt.py - on passe a la forme standard (3 args)
    _patch_file(src_emo / "model_prompt.py", [(
        r"entity_embeds = self\.kg_encoder\(node_embeds, self\.edge_index\) \+ node_embeds.*?\n\s*#entity_embeds = self\.kg_encoder\(node_embeds, self\.edge_index, self\.edge_type\) \+ node_embeds",
        "entity_embeds = self.kg_encoder(node_embeds, self.edge_index, self.edge_type) + node_embeds",
    )], description="model_prompt.py RGCNConv call")

    # Patch 6 : Accelerator(fp16=...) retire en accelerate 0.12+
    #           On s'appuie sur `accelerate launch --mixed_precision <mp>` pour la config.
    accel_patch = [(
        r"accelerator = Accelerator\(device_placement=False, fp16=args\.fp16\)",
        "accelerator = Accelerator(device_placement=False)  # PATCH: mixed_precision via `accelerate launch`",
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, accel_patch, description=f"Accelerator fp16 kwarg ({name})")

    # Patch 7 : seed_torch dans config.py
    #   - torch.set_deterministic -> torch.use_deterministic_algorithms (retire en torch 2.0)
    #   - CUDA_LAUNCH_BLOCKING=1 et CUBLAS_WORKSPACE_CONFIG:4096:8 : residus debug qui
    #     serialisent les kernels CUDA et coutent 5-10x en wall-time sur GPU moderne.
    _patch_file(src_emo / "config.py", [
        (
            r"torch\.set_deterministic\(True\)",
            "torch.use_deterministic_algorithms(True, warn_only=True)  # PATCH: set_deterministic retire en torch 2.0",
        ),
        (
            r'^(\s*)os\.environ\["CUDA_LAUNCH_BLOCKING"\] = "1"\s*$',
            r'\1# os.environ["CUDA_LAUNCH_BLOCKING"] = "1"  # PATCH: desactive (5-10x plus lent sur GPU moderne)',
        ),
        (
            r'^(\s*)os\.environ\["CUBLAS_WORKSPACE_CONFIG"\] = ":4096:8"\s*$',
            r'\1# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"  # PATCH: desactive (lie a determinisme strict)',
        ),
    ], description="config.py seed_torch")

    # Patch 8 : accelerator.use_fp16 retire en accelerate 0.20
    #   Les collators lisent ce flag pour activer `torch.cuda.amp.autocast` manuellement ;
    #   comme on passe --mixed_precision bf16 via `accelerate launch`, l'autocast est deja
    #   gere et on peut renvoyer False (fallback sur bf16 automatique).
    use_fp16_patch = [(
        r"accelerator\.use_fp16",
        "getattr(accelerator, 'use_fp16', False)",
    )]
    for name in ("train_pre.py", "train_emp.py", "infer_emp.py"):
        _patch_file(src_emo / name, use_fp16_patch, description=f"accelerator.use_fp16 ({name})")

    # Patch 9 : .cuda() hardcode dans dataset_dbpedia.py
    #   Tolere un fallback CPU si aucun GPU n'est detecte au moment de construire DBpedia
    #   (cas transitoire : process zombie tenant de la VRAM, driver en power-save, ...).
    #   RGCNConv gere ensuite le deplacement vers le device du modele lors du forward.
    _patch_file(src_emo / "dataset_dbpedia.py", [
        (
            r"^([ \t]+)self\.edge_index = edge\[:, :2\]\.t\(\)\.cuda\(\)$",
            r"\1_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n\1self.edge_index = edge[:, :2].t().to(_dev)  # PATCH: tolere CPU",
        ),
        (
            r"^([ \t]+)self\.edge_type = edge\[:, 2\]\.cuda\(\)$",
            r"\1self.edge_type = edge[:, 2].to(_dev)  # PATCH: tolere CPU",
        ),
    ], description="dataset_dbpedia.py .cuda() hardcode")

    # Patch 10 : prints de debug `print("here")` laisses par les auteurs
    #   dataset_emp.py (2 occurrences), imdb_review_entity_filter.py, train_pre.py, co_appear.py
    debug_patch = [(
        r"^(\s*)print\(([\"'])here\2\)\s*$",
        r'\1pass  # PATCH: debug print("here") supprime',
    )]
    for name in ("dataset_emp.py", "imdb_review_entity_filter.py", "train_pre.py", "co_appear.py"):
        _patch_file(src_emo / name, debug_patch, description=f'debug print("here") dans {name}')

    # Patch 11 : ensure save/redial_rec existe avant ecriture rec.json
    _patch_file(src_emo / "evaluate_rec.py", [(
        r'^(\s*)self\.log_file = open\("save/redial_rec/rec\.json", ["\']w["\'], buffering=1\)\s*$',
        r'\1import os\n\1os.makedirs("save/redial_rec", exist_ok=True)  # PATCH: cree le dossier de logs\n\1self.log_file = open("save/redial_rec/rec.json", "w", buffering=1)',
    )], description="evaluate_rec.py rec.json directory")

    # Patch 12 : transformers.file_utils -> transformers.utils (preemptif)
    #   file_utils est un alias deprecie depuis 4.25 et retire en 5.0.
    #   Ca marche encore en 4.57 mais emet un FutureWarning.
    _patch_file(src_emo / "model_gpt2.py", [(
        r"from transformers\.file_utils import ModelOutput",
        "from transformers.utils import ModelOutput  # PATCH: file_utils deprecated -> utils",
    )], description="file_utils -> utils (model_gpt2.py)")

    # Patch 13 : PromptGPT2forCRS doit heriter de GenerationMixin (transformers >= 4.50)
    #   Sans ca, model.generate() leve AttributeError dans infer_emp.py / train_emp.
    #   Idempotent : on detecte si l'import existe deja avant de l'injecter.
    gpt2_path = src_emo / "model_gpt2.py"
    if gpt2_path.exists() and "from transformers.generation import GenerationMixin" not in gpt2_path.read_text():
        _patch_file(gpt2_path, [
            (
                r"^from transformers\.modeling_outputs import[^\n]*\n",
                r"\g<0>from transformers.generation import GenerationMixin  # PATCH: generate() post transformers 4.50\n",
            ),
        ], description="GenerationMixin import (model_gpt2.py)")
    _patch_file(gpt2_path, [
        (
            r"class\s+PromptGPT2forCRS\s*\(\s*GPT2PreTrainedModel\s*\)\s*:",
            "class PromptGPT2forCRS(GPT2PreTrainedModel, GenerationMixin):  # PATCH: GenerationMixin",
        ),
    ], description="PromptGPT2forCRS heritage (model_gpt2.py)")

    # Patch 14 : train_rec.py ne doit plus forcer CUDA_VISIBLE_DEVICES=3
    #   L'auteur du depot avait une machine multi-GPU ; sur un setup mono-GPU
    #   ca rend CUDA invisible pour Accelerate qui bascule en CPU silencieusement.
    _patch_file(src_emo / "train_rec.py", [(
        r'^\s*os\.environ\[\s*["\']CUDA_VISIBLE_DEVICES["\']\s*\]\s*=\s*["\']3["\']\s*$',
        r'# os.environ["CUDA_VISIBLE_DEVICES"] = "3"  # PATCH: retire (forcait GPU 3)',
    )], description="train_rec.py CUDA_VISIBLE_DEVICES")

    # Patch 15 : log/ doit exister avant `logger.add("log/<fichier>")` (loguru)
    #   Sinon FileNotFoundError au demarrage. Idempotent : on skip si le makedirs
    #   est deja present dans le fichier.
    log_mkdir_patch = [(
        r"^(\s*)logger\.remove\(\)",
        r'\1os.makedirs("log", exist_ok=True)  # PATCH: cree log/ avant logger.add\n\1logger.remove()',
    )]
    for name in ("train_pre.py", "train_rec.py", "train_emp.py", "infer_emp.py"):
        p = src_emo / name
        if p.exists() and 'os.makedirs("log", exist_ok=True)' not in p.read_text():
            _patch_file(p, log_mkdir_patch, description=f"log/ mkdir ({name})")
        else:
            print(f"[patch] log/ mkdir ({name}): deja applique (idempotent)")

    # Patch 16 : chemins de sortie stabilises (pas de suffixe timestamp)
    #   train_pre.py ajoutait `args.output_dir = args.output_dir + local_time` ce
    #   qui produisait `data/saved/pre_2026-04-18-22-15` et faisait planter
    #   train_rec.py (qui cherche `data/saved/pre/model.pt`). Le code amont
    #   varie : train_emp.py ecrit `args.output_dir+local_time` (sans espaces)
    #   alors que train_pre.py/train_rec.py utilisent `+ local_time` avec espaces.
    #   Le regex supporte les deux formats.
    stable_out = [(
        r"^(\s*)args\.output_dir\s*=\s*args\.output_dir\s*\+\s*local_time\s*$",
        r"\1# args.output_dir = args.output_dir + local_time  # PATCH: chemin stable",
    )]
    for name in ("train_pre.py", "train_emp.py", "train_rec.py"):
        _patch_file(src_emo / name, stable_out, description=f"{name} chemin sortie stable")

    # Patch 17 : train_pre.py --output_dir default -> data/saved/pre
    _patch_file(src_emo / "train_pre.py", [(
        r'(--output_dir[^\n]*default\s*=\s*)["\'][^"\']*["\']',
        r'\1"data/saved/pre"',
    )], description="train_pre.py --output_dir default")

    # Patch 18 : train_emp.py --output_dir default -> data/saved/emp
    _patch_file(src_emo / "train_emp.py", [(
        r'(--output_dir[^\n]*default\s*=\s*)["\'][^"\']*["\']',
        r'\1"data/saved/emp"',
    )], description="train_emp.py --output_dir default")

    # Patch 19 : infer_emp.py doit charger LOCALEMENT si --model est un dossier
    #   Sinon HFValidationError: "Repo id must be..." quand on passe data/saved/emp/.
    #   Le code amont utilise `PromptGPT2forCRS.from_pretrained(args.model)` et NON
    #   `AutoModelForCausalLM.from_pretrained`. Le regex est adapte en consequence.
    _patch_file(src_emo / "infer_emp.py", [(
        r"(PromptGPT2forCRS\.from_pretrained\()(args\.model)(\s*\))",
        r"\1\2, local_files_only=os.path.isdir(args.model)\3  # PATCH: dossier local",
    )], description="infer_emp.py local_files_only")

    # Patch 20 : retain_graph=True retire (fuite memoire + pas necessaire)
    retain_patch = [(
        r"accelerator\.backward\(\s*loss\s*,\s*retain_graph\s*=\s*True\s*\)",
        "accelerator.backward(loss)  # PATCH: retain_graph=True retire",
    )]
    for name in ("train_pre.py", "train_rec.py"):
        _patch_file(src_emo / name, retain_patch, description=f"retain_graph retire ({name})")

    # Patch 22 : DataLoader num_workers + pin_memory dans train_pre.py et train_rec.py
    #   Le code amont parse --num_workers mais ne le passe PAS au DataLoader dans
    #   ces 2 scripts. Sans workers paralleles, le pipeline CPU->GPU est le
    #   goulot (20-30% de speedup a gagner sur 9950X3D). On ajoute aussi
    #   pin_memory=True pour accelerer le transfert host->device.
    #   Idempotent : on skip si num_workers=args.num_workers est deja present.
    dl_patch = [(
        r"(DataLoader\(\s*\w+,\s*batch_size=args\.per_device_(?:train|eval)_batch_size,\s*)(collate_fn=data_collator,?)\s*((?:shuffle=True)?)\s*\)",
        r"\1num_workers=args.num_workers, pin_memory=True, \2 \3)  # PATCH 22: DataLoader workers",
    )]
    for name in ("train_pre.py", "train_rec.py"):
        p = src_emo / name
        if not p.exists():
            print(f"[patch] Patch 22 SKIP {name}: fichier absent")
            continue
        if "num_workers=args.num_workers" in p.read_text():
            print(f"[patch] Patch 22 ({name}): deja applique (idempotent)")
            continue
        _patch_file(p, dl_patch, description=f"Patch 22 DataLoader workers ({name})")

    # Patch 23 : torch.compile(prompt_encoder) apres accelerator.prepare dans train_rec.py
    #   Gain attendu : 1.2-1.5x wall-time apres le warmup de compilation (~30s).
    #   Idempotent : on skip si torch.compile est deja present dans le fichier.
    #   Wrap dans try/except pour tolerer un echec de compilation (RGCNConv peut
    #   ne pas aimer certains patterns dynamiques).
    compile_patch = [(
        r"(prompt_encoder, optimizer, train_dataloader, valid_dataloader, test_dataloader\s*=\s*accelerator\.prepare\(\s*\n?\s*prompt_encoder, optimizer, train_dataloader, valid_dataloader, test_dataloader\s*\n?\s*\))",
        r"\1\n    try:\n        prompt_encoder = torch.compile(prompt_encoder, mode='reduce-overhead')  # PATCH 23: torch.compile\n        logger.info('[PATCH 23] torch.compile(prompt_encoder) OK')\n    except Exception as _e:\n        logger.warning(f'[PATCH 23] torch.compile fallback eager: {_e}')",
    )]
    p = src_emo / "train_rec.py"
    if p.exists() and "torch.compile" not in p.read_text():
        _patch_file(p, compile_patch, description="Patch 23 torch.compile(prompt_encoder)")
    else:
        print("[patch] Patch 23 (train_rec.py): deja applique (idempotent) ou fichier absent")

    # Patch 21 : assertions idempotentes - detectent si une regex a cesse de matcher
    #   (ex. suite a un rebase upstream). On ne raise pas : on avertit seulement.
    def _check(path, needle, desc):
        if not path.exists():
            print(f"[assert] SKIP {desc}: {path} absent")
            return
        if needle in path.read_text():
            print(f"[assert] OK   {desc}")
        else:
            print(f"[assert] KO   {desc} (patch a verifier manuellement)")

    _check(src_emo / "train_rec.py", "from torch.optim import AdamW", "Patch 1 AdamW")
    _check(src_emo / "model_gpt2.py", "transformers.pytorch_utils", "Patch 2 pruneable_heads")
    _check(src_emo / "model_prompt.py", "self.edge_type", "Patch 5 RGCNConv edge_type")
    _check(src_emo / "config.py", "use_deterministic_algorithms", "Patch 7 determinisme")
    _check(src_emo / "model_gpt2.py", "GenerationMixin", "Patch 13 GenerationMixin")
    _check(src_emo / "train_rec.py", 'os.makedirs("log"', "Patch 15 log/ mkdir")
    _check(src_emo / "infer_emp.py", "local_files_only=os.path.isdir", "Patch 19 local_files_only")
    _check(src_emo / "train_rec.py", "num_workers=args.num_workers", "Patch 22 DataLoader workers")
    _check(src_emo / "train_rec.py", "torch.compile", "Patch 23 torch.compile")

In [ ]:
# Cellule 5 - Telechargement des archives emo_data / ckpt depuis Google Drive
# gdown est necessaire : Google Drive sert une page HTML d'interstitiel pour
# tout fichier > 100 MB (ckpt.zip pese 679 MB). curl recupererait alors le
# HTML au lieu du .zip. gdown detecte cette page et suit le token `confirm`.
import gdown


def download_gdrive(file_id, dst, description):
    """Telecharge une archive depuis Google Drive (gere l'interstitiel > 100 MB)."""
    dst = Path(dst)
    if dst.exists() and dst.stat().st_size > 1024:
        print(f"[{description}] deja present: {dst} ({dst.stat().st_size / 1e6:.1f} MB)")
        return dst
    dst.parent.mkdir(parents=True, exist_ok=True)
    print(f"[{description}] gdown download (id={file_id}) -> {dst}")
    try:
        out = gdown.download(id=file_id, output=str(dst), quiet=False)
    except Exception as exc:
        print(f"[{description}] ECHEC gdown: {exc}")
        return None
    if out is None or not dst.exists() or dst.stat().st_size < 1024:
        print(f"[{description}] telechargement vide, deposer manuellement dans: {dst}")
        if dst.exists():
            dst.unlink(missing_ok=True)
        return None
    print(f"[{description}] OK ({dst.stat().st_size / 1e6:.1f} MB)")
    return dst


def unzip_if_needed(archive, dst_dir):
    """Decompresse l'archive si necessaire."""
    dst_dir = Path(dst_dir)
    if dst_dir.exists() and any(dst_dir.iterdir()):
        print(f"  {dst_dir} deja extrait.")
        return dst_dir
    if archive is None or not Path(archive).exists():
        print(f"  archive manquante, extraction ignoree.")
        return None
    dst_dir.mkdir(parents=True, exist_ok=True)
    result = subprocess.run(
        ["unzip", "-oq", str(archive), "-d", str(dst_dir)],
        capture_output=True,
        text=True,
        check=False,
    )
    if result.returncode != 0:
        print("[STDERR]", result.stderr)
    print(f"  extrait -> {dst_dir}")
    return dst_dir


def download_external_assets(cfg):
    """Telecharge + extrait emo_data.zip et ckpt.zip depuis Google Drive."""
    if cfg["skip_download"]:
        print("skip_download=True -> archives non recuperees.")
        return
    download_gdrive(cfg["emo_data_gdrive_id"], cfg["emo_data_archive"], "emo_data")
    unzip_if_needed(cfg["emo_data_archive"], cfg["emo_data_dir"])
    download_gdrive(cfg["ckpt_gdrive_id"], cfg["ckpt_archive"], "ckpt")
    unzip_if_needed(cfg["ckpt_archive"], cfg["ckpt_dir"])

### 2bis. Modeles de base HF + poids pre-entraines (optionnel)

Les scripts ECR chargent DialoGPT-small et RoBERTa-base via des chemins **locaux** (`save/dialogpt/`, `save/roberta/`) au sein de `src_emo/`. On les telecharge une fois via `huggingface_hub.snapshot_download` — les caches Docker Compose (`hf_cache`) evitent tout re-download entre runs.

`install_pretrained_ckpts` est active **uniquement** si le flag `use_pretrained_ckpt=True`. Il copie :

- `data/ckpt/pre/`  -> `src_emo/data/saved/pre/`   (Emotional Semantic Fusion)
- `data/ckpt/rec/`  -> `src_emo/data/saved/rec/`   (Emotion-aware Recommendation)
- `data/ckpt/emp/`  -> `src_emo/data/saved/emp/`   (Emotion-aligned DialoGPT)

Ainsi `train_rec.py` (qui lit `--prompt_encoder data/saved/pre/`) et `infer_emp.py` (qui lit `--model data/saved/emp/`) trouvent les poids directement. Les trois cellules `run_*` respectent ce flag : elles sautent proprement le training correspondant. Passer `use_pretrained_ckpt=False` relance le pipeline complet sans code a modifier.

In [ ]:
# Cellule 5b - Modeles de base HF + poids pre-entraines (optionnel)
# Les scripts ECR appellent `AutoTokenizer.from_pretrained("save/dialogpt/")` et
# `AutoModel.from_pretrained("save/roberta/")` : ces repertoires doivent exister
# dans src_emo/. On telecharge depuis HF Hub une fois (cache volume HF partage).
from huggingface_hub import snapshot_download


def install_base_models(cfg):
    """Telecharge DialoGPT-small + RoBERTa-base dans src_emo/save/ (HF snapshots)."""
    src_emo = cfg["src_emo_dir"]
    for hf_id, rel_dir in cfg["base_models"].items():
        local = src_emo / rel_dir
        if (local / "config.json").exists():
            print(f"[base] {hf_id} deja present: {local}")
            continue
        local.mkdir(parents=True, exist_ok=True)
        print(f"[base] snapshot_download {hf_id} -> {local}")
        try:
            snapshot_download(
                repo_id=hf_id,
                local_dir=str(local),
                local_dir_use_symlinks=False,
            )
        except TypeError:
            # API huggingface_hub >= 0.23 a retire local_dir_use_symlinks
            snapshot_download(repo_id=hf_id, local_dir=str(local))
        print(f"[base] {hf_id} OK ({sum(1 for _ in local.iterdir())} fichiers)")


def install_pretrained_ckpts(cfg):
    """Copie data/ckpt/{pre,rec,emp}/* dans src_emo/data/saved/{pre,rec,emp}/.

    Active seulement si cfg["use_pretrained_ckpt"] est True.
    Permet aux scripts train_rec.py (qui lit --prompt_encoder data/saved/pre/)
    et infer_emp.py (qui lit --model data/saved/emp/) de trouver les poids sans
    re-entrainement. Aucun fichier n'est supprime : on peut toujours repartir
    sur un training from-scratch en desactivant le flag.
    """
    if not cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=False -> poids fournis non installes (training complet).")
        return
    src_emo = cfg["src_emo_dir"]
    ckpt = cfg["ckpt_dir"]
    saved = src_emo / "data" / "saved"
    saved.mkdir(parents=True, exist_ok=True)
    for sub in ("pre", "rec", "emp"):
        src = ckpt / sub
        dst = saved / sub
        if not src.exists():
            print(f"[ckpt] source absente: {src}")
            continue
        dst.mkdir(parents=True, exist_ok=True)
        for f in src.iterdir():
            if f.name.startswith(".") or f.name == "__MACOSX":
                continue
            target = dst / f.name
            if target.exists():
                continue
            if f.is_file():
                shutil.copy2(f, target)
            else:
                shutil.copytree(f, target)
        print(f"[ckpt] {src} -> {dst}")

## 3. Pretraitement du dataset ReDial (Section 4.1)

Le README du depot ECR enchaine les etapes suivantes :

```bash
cd src_emo
cp -r data/emo_data/* data/redial/
python data/redial/process.py
```

`process.py` integre les annotations GPT-3.5 (9 labels d'emotion : *like, curious, happy, grateful, negative, neutral, nostalgia, agreement, surprise* — cf. Section 4.1.1) au corpus ReDial brut et fournit les entites DBpedia alignees. Pour la recommandation, `process_mask.py` masque l'item cible ; `merge.py` recolte les predictions du modele conversationnel UniCRS. La fonction ci-dessous orchestre ces commandes.

In [ ]:
# Cellule 6 - Pretraitement du dataset ReDial (Section 4.1)
#
# Patch post-conversation : la version avec `capture_output=True` est remplacee
# par un streaming ligne-a-ligne + tee dans `logs/<description>.log`. Sans ca,
# `accelerate launch train_rec.py` ne retourne aucune sortie pendant 2h et on
# ne peut pas detecter un plantage ou extraire les metriques en temps reel.
def _run(cmd, cwd=None, description="", log_dir=None):
    """Lance une commande en streaming stdout/stderr + tee vers `logs/<desc>.log`.

    Le retour booleen (True si exit 0) reste compatible avec l'ancienne API.
    """
    log_dir = Path(log_dir) if log_dir else (ROOT / "logs")
    log_dir.mkdir(parents=True, exist_ok=True)
    safe_desc = (description or "cmd").replace(" ", "_").replace("/", "_")
    log_path = log_dir / f"{safe_desc}.log"
    print(f"[run] {description} :: {' '.join(str(x) for x in cmd)}")
    print(f"[run] log -> {log_path}")

    with log_path.open("w") as log_f:
        log_f.write(f"# cmd: {' '.join(str(x) for x in cmd)}\n")
        log_f.write(f"# cwd: {cwd}\n\n")
        log_f.flush()
        try:
            proc = subprocess.Popen(
                cmd,
                cwd=cwd,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                bufsize=1,
            )
        except FileNotFoundError as exc:
            print(f"[run] commande introuvable: {exc}")
            return False
        for line in proc.stdout:
            print(line, end="", flush=True)
            log_f.write(line)
            log_f.flush()
        ret = proc.wait()
    if ret != 0:
        print(f"[run] FAIL {description} exit={ret} (voir {log_path})")
    else:
        print(f"[run] OK {description}")
    return ret == 0


def prepare_redial_data(cfg):
    """Reproduit la preparation `data/redial/` et `data/redial_gen/` du README ECR."""
    src_emo = cfg["src_emo_dir"]
    if not src_emo.exists():
        print(f"src_emo indisponible: {src_emo} -> clone le depot d'abord.")
        return False
    redial = src_emo / "data" / "redial"
    redial_gen = src_emo / "data" / "redial_gen"
    emo = cfg["emo_data_dir"]

    redial.mkdir(parents=True, exist_ok=True)
    redial_gen.mkdir(parents=True, exist_ok=True)

    if emo.exists():
        for p in emo.iterdir():
            tgt = redial / p.name
            if not tgt.exists():
                (shutil.copy2 if p.is_file() else shutil.copytree)(p, tgt)
    else:
        print(f"emo_data non trouve ({emo}) - les scripts supposent qu'il a ete decompresse.")

    ok = True
    if cfg["dry_run"]:
        print("DRY_RUN=True -> scripts process.py / process_mask.py / merge.py non executes.")
        return True

    # Les scripts ECR utilisent des chemins relatifs (ex: open('entity2id.json')) :
    # ils doivent etre lances depuis leur propre repertoire, pas depuis src_emo.
    ok &= _run(["python", "process.py"],      cwd=redial,     description="process.py")
    ok &= _run(["python", "process_mask.py"], cwd=redial,     description="process_mask.py")
    if redial.exists() and redial_gen.exists():
        for p in redial.iterdir():
            tgt = redial_gen / p.name
            if not tgt.exists():
                (shutil.copy2 if p.is_file() else shutil.copytree)(p, tgt)
    ok &= _run(["python", "merge.py"],        cwd=redial_gen, description="merge.py")
    return ok

## 4. Sous-tache Emotional Semantic Fusion (Section 4.1)

La fusion emotionnelle est entrainee avec `train_pre.py`. L'objectif est de fusionner la semantique des mots (token-level) et des entites emotionnelles (utterance-level) avant la phase de recommandation. Les hyperparametres du depot sont :

- `num_train_epochs=10`, `gradient_accumulation_steps=4`, `per_device_train_batch_size=16` ;
- `max_length=200`, `prompt_max_length=200`, `entity_max_length=32` ;
- `learning_rate=5e-4`, `num_warmup_steps=1389`.

L'option `--nei_mer` active la fusion emotion/neighbour decrite Section 4.1.

In [ ]:
# Cellule 7 - Emotional Semantic Fusion (train_pre.py)
def run_emotional_semantic_fusion(cfg):
    """Execute `accelerate launch train_pre.py ...` (Section 4.1)."""
    if cfg["dry_run"]:
        print("DRY_RUN=True -> train_pre.py non lance (>6h sur GPU 24GB).")
        return None
    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_pre.py ignore (poids charges depuis data/ckpt/pre/).")
        return True
    # Article: per_device_batch=16, grad_accum=4 -> effective batch 64.
    batch, grad_accum = scaled_batch(cfg, per_device_batch=16, grad_accum=4)
    cmd = accelerate_launch_cmd(cfg, "train_pre.py") + [
        "--dataset", "redial",
        "--num_train_epochs", "10",
        "--gradient_accumulation_steps", str(grad_accum),
        "--per_device_train_batch_size", str(batch),
        "--per_device_eval_batch_size", "64",
        "--num_warmup_steps", "1389",
        "--max_length", "200",
        "--prompt_max_length", "200",
        "--entity_max_length", "32",
        "--learning_rate", "5e-4",
        "--seed", str(cfg["seed"]),
        "--num_workers", str(cfg.get("dataloader_num_workers", 8)),  # Patch 22 optim
        "--nei_mer",
    ]
    return _run(cmd, cwd=cfg["src_emo_dir"], description="train_pre")

## 5. Sous-tache Emotion-aware Item Recommendation (Section 4.2)

Cette etape entraine le module de recommandation avec :

- representations locales / globales emotion-aware (Eq. 3-6) ;
- **feedback-aware item reweighting** (Eq. 7) avec les poids `like=2.0`, `dislike=1.0`, `not say=0.5` ;
- option `--use_sentiment` pour utiliser la classification fine-tunee GPT-2 du README.

In [ ]:
# Cellule 8 - Emotion-aware Item Recommendation (train_rec.py)
#
# La version mono-run est conservee pour retro-compat, mais le pipeline principal
# utilise `run_recommendation_sweep` qui enchaine les 5 variantes de l'article :
#
#   +--------+----------+----------------+----------------+---------------+
#   | Nom    | nei_mer  | weighted_loss  | use_sentiment  | global entity |
#   +--------+----------+----------------+----------------+---------------+
#   | UniCRS |   OFF    |      OFF       |      OFF       |     OFF       |
#   | ECR[L] |   ON     |      OFF       |      ON        |     OFF       |
#   | ECR[LS]|   ON     |      ON        |      ON        |     OFF       |
#   | ECR[LG]|   ON     |      OFF       |      ON        |     ON        |
#   | ECR    |   ON     |      ON        |      ON        |     ON        |
#   +--------+----------+----------------+----------------+---------------+
#
# Chaque variante produit :
#   - `logs/train_rec_<name>.log`  -> parse pour Table 1/3 + loss curve
#   - `data/saved/rec_<name>/`     -> ckpt (le dernier ecrase save/redial_rec/rec.json
#                                     si on genere aussi les outputs pour Table 2)
#
# NB : apres inspection du depot amont, le flag "global entity" n'existe PAS
#      sous un argparse dedie. La dimension globale (co-occurrence TOCoAppear)
#      est activee par `--nei_mer` dans KGPrompt.forward (cf. model_prompt.py).

# REC_VARIANTS : mapping reel apres inspection de `train_rec.py` amont (voir
# https://github.com/zxd-octopus/ECR/blob/main/src_emo/train_rec.py et
# model_prompt.py::KGPrompt.forward).
#
# Correspondance flags <-> theorie (article Section 4.2 + 6.1) :
#   --use_sentiment : active l'utilisation du classifier d'emotion GPT-2 et
#                     la fusion LOCALE emotion-entity (Eq. 3-4).
#   --nei_mer       : active la fusion GLOBALE via co-occurrence TOCoAppear
#                     (get_nei_rep(nei_mvs)), equivalent a E'_g de l'Eq. 6.
#   --weighted_loss : active le feedback-aware item reweighting (Eq. 7).
#
# Les 5 variantes de Table 3 :
#   UniCRS : aucune emotion (fallback le plus proche : tous les flags off).
#            Le code ECR ne permet pas de desactiver completement emo fusion,
#            donc ce run sert de baseline quasi-UniCRS ; la ligne Table 1 reste
#            marquee "proche UniCRS" (vraie reproduction necessite le depot
#            original UniCRS separe - hors budget).
#   ECR[L] : local emotion uniquement.
#   ECR[LS]: local + reweighting.
#   ECR[LG]: local + global co-appear.
#   ECR    : tout (local + reweighting + global).
REC_VARIANTS = [
    {"name": "UniCRS", "nei_mer": False, "weighted_loss": False, "use_sentiment": False},
    {"name": "ECR_L",  "nei_mer": False, "weighted_loss": False, "use_sentiment": True},
    {"name": "ECR_LS", "nei_mer": False, "weighted_loss": True,  "use_sentiment": True},
    {"name": "ECR_LG", "nei_mer": True,  "weighted_loss": False, "use_sentiment": True},
    {"name": "ECR",    "nei_mer": True,  "weighted_loss": True,  "use_sentiment": True},
]


def _build_train_rec_cmd(cfg, variant):
    """Construit `accelerate launch train_rec.py ...` pour une variante ablation.

    Les flags optionnels `--nei_mer` / `--weighted_loss` / `--use_sentiment`
    correspondent exactement aux champs de `variant` (verifie dans le code
    amont `src_emo/train_rec.py`). Aucun autre flag "global" n'existe : la
    dimension globale de Section 4.2.1 est activee par `--nei_mer`.

    On passe aussi `--num_workers 8` pour l'optimisation DataLoader (necessite
    le Patch 22 qui injecte num_workers dans le DataLoader du script amont).
    """
    batch, grad_accum = scaled_batch(cfg, per_device_batch=cfg["batch_rec"] // 8, grad_accum=8)
    cmd = accelerate_launch_cmd(cfg, "train_rec.py") + [
        "--dataset", "redial_gen",
        "--n_prefix_rec", "10",
        "--num_train_epochs", "5",
        "--per_device_train_batch_size", str(batch),
        "--per_device_eval_batch_size", "32",
        "--gradient_accumulation_steps", str(grad_accum),
        "--num_warmup_steps", "530",
        "--context_max_length", "200",
        "--prompt_max_length", "200",
        "--entity_max_length", "32",
        "--learning_rate", str(cfg["lr_dialogpt"]),
        "--seed", str(cfg["seed_rec_runs"]),
        "--like_score", str(cfg["feedback_weights"]["like"]),
        "--dislike_score", str(cfg["feedback_weights"]["dislike"]),
        "--notsay_score", str(cfg["feedback_weights"]["not say"]),
        "--num_workers", str(cfg.get("dataloader_num_workers", 8)),
    ]
    if variant["nei_mer"]:
        cmd.append("--nei_mer")
    if variant["weighted_loss"]:
        cmd.append("--weighted_loss")
    if variant["use_sentiment"]:
        cmd.append("--use_sentiment")
    return cmd


def run_recommendation_training(cfg):
    """Retro-compat : lance UNE seule variante (ECR complete)."""
    if cfg["dry_run"]:
        print("DRY_RUN=True -> train_rec.py non lance (entrainement GPU necessaire).")
        return None
    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_rec.py ignore (poids charges depuis data/ckpt/rec/).")
        return True
    pre_ckpt = cfg["src_emo_dir"] / "data" / "saved" / "pre" / "model.pt"
    if not pre_ckpt.exists():
        print(f"[skip] train_rec: checkpoint pre absent ({pre_ckpt}).")
        return False
    cmd = _build_train_rec_cmd(cfg, REC_VARIANTS[-1])  # ECR complete
    return _run(cmd, cwd=cfg["src_emo_dir"], description="train_rec")


def run_recommendation_sweep(cfg):
    """Boucle sur les 5 variantes Table 1/3 (UniCRS, ECR[L], ECR[LS], ECR[LG], ECR).

    Chaque run prend ~1.5-2h sur RTX 5090 bf16 eff_batch=128 -> ~10h total.
    Les resultats sont ecrits dans `logs/train_rec_<name>.log` et parses par
    `src.ecr.metrics_parser.write_rec_results` dans une cellule ulterieure.

    Note : contrairement a `run_recommendation_training`, CE sweep entraine
    les 5 variantes meme si `use_pretrained_ckpt=True`. L'archive fournie ne
    contient qu'un ckpt `rec/` pour la variante ECR complete - elle ne
    permet pas de reproduire l'etude d'ablation Table 3. Le ckpt `pre/` est
    quand meme utilise comme point de depart pour les 5 runs.
    """
    if cfg["dry_run"]:
        print("DRY_RUN=True -> sweep recommendation ignore.")
        return None
    if not cfg.get("run_rec_sweep"):
        print("run_rec_sweep=False -> sweep desactive, fallback article pour Tables 1/3.")
        return None
    pre_ckpt = cfg["src_emo_dir"] / "data" / "saved" / "pre" / "model.pt"
    if not pre_ckpt.exists():
        print(f"[skip] sweep: checkpoint pre absent ({pre_ckpt}). Active use_pretrained_ckpt ou relance train_pre.")
        return None

    results = {}
    for variant in REC_VARIANTS:
        name = variant["name"]
        cmd = _build_train_rec_cmd(cfg, variant)
        ok = _run(cmd, cwd=cfg["src_emo_dir"], description=f"train_rec_{name}")
        results[name] = ok
        if not ok:
            print(f"[sweep] {name} a echoue -> on continue avec les variantes suivantes.")
    print("\n=== Recap sweep ===")
    for k, v in results.items():
        print(f"  {k:8s}: {'OK' if v else 'FAIL'}")
    return results

## 6. Sous-tache Emotion-aligned Response Generation (Section 4.3)

Deux variantes sont fournies dans le depot ECR :

- **ECR[DialoGPT]** : `train_emp.py` puis `infer_emp.py`. On injecte le prompt emotion-aligned (Eq. 8) construit a partir des reviews IMDb filtrees (34,953 reviews / 4,092 films).
- **ECR[Llama 2-Chat]** : fine-tuning LoRA via LLaMA Board sur `src_emo/data/emo_data/llama_train.json` + `llama_test.json` (2,459 reviews / 1,553 films).

La fonction ci-dessous pilote la variante DialoGPT (la variante Llama est fine-tunee via un outil externe).

In [ ]:
# Cellule 9 - Emotion-aligned Response Generation (DialoGPT)
def run_response_generation_training(cfg):
    if cfg["dry_run"]:
        print("DRY_RUN=True -> pipeline generation non lance.")
        return None
    src_emo = cfg["src_emo_dir"]
    redial = src_emo / "data" / "redial"
    redial_gen = src_emo / "data" / "redial_gen"
    ok = True

    # merge_rec.py lit ../../save/redial_rec/rec.json -> cwd doit etre redial_gen.
    rec_json = src_emo / "save" / "redial_rec" / "rec.json"
    if not rec_json.exists():
        print(f"[skip] merge_rec: fichier absent ({rec_json}). train_rec n'a pas produit de recommandations.")
        return False
    ok &= _run(["python", "merge_rec.py"],              cwd=redial_gen, description="merge_rec")
    # imdb_review_entity_filter.py importe dataset_dbpedia -> cwd = src_emo.
    ok &= _run(["python", "imdb_review_entity_filter.py"], cwd=src_emo,   description="imdb_review_entity_filter")
    # process_empthetic.py lit movie_reviews_entities_filtered.json en local -> cwd = redial.
    ok &= _run(["python", "process_empthetic.py"],      cwd=redial,     description="process_empthetic")

    if cfg.get("use_pretrained_ckpt"):
        print("use_pretrained_ckpt=True -> train_emp.py ignore, on passe direct a l'inference.")
    else:
        # Article: per_device_batch=20, grad_accum=1 -> effective batch 20.
        batch, grad_accum = scaled_batch(cfg, per_device_batch=20, grad_accum=1)
        train_cmd = accelerate_launch_cmd(cfg, "train_emp.py") + [
            "--dataset", "redial",
            "--num_train_epochs", "15",
            "--gradient_accumulation_steps", str(grad_accum),
            "--ignore_pad_token_for_loss",
            "--per_device_train_batch_size", str(batch),
            "--per_device_eval_batch_size", "64",
            "--num_warmup_steps", "9965",
            "--context_max_length", "150",
            "--resp_max_length", "150",
            "--learning_rate", str(cfg["lr_dialogpt"]),
            "--num_workers", str(cfg.get("dataloader_num_workers", 8)),  # Patch 22 optim (amont support natif)
        ]
        ok &= _run(train_cmd, cwd=src_emo, description="train_emp")

    if cfg.get("run_infer_emp", True):
        infer_cmd = accelerate_launch_cmd(cfg, "infer_emp.py") + [
            "--dataset", "redial_gen",
            "--split", "test",
            "--per_device_eval_batch_size", "256",
            "--context_max_length", "150",
            "--resp_max_length", "150",
            "--num_workers", str(cfg.get("dataloader_num_workers", 8)),  # Patch 22 optim
        ]
        ok &= _run(infer_cmd, cwd=src_emo, description="infer_emp")

        # On exporte aussi la sortie dans results/generations/ecr_dialogpt.jsonl
        # pour que le scorer Qwen puisse la noter aux cotes de Llama 2.
        src_json = src_emo / "save" / "redial_gen" / "output_test.json"
        dst_json = cfg["generations_dir"] / "ecr_dialogpt.jsonl"
        if src_json.exists():
            dst_json.parent.mkdir(parents=True, exist_ok=True)
            import json as _json
            try:
                data = _json.loads(src_json.read_text())
                with dst_json.open("w") as out:
                    for i, row in enumerate(data if isinstance(data, list) else []):
                        out.write(_json.dumps({
                            "dialogue_id": row.get("dialogue_id", i),
                            "history": row.get("context", row.get("history", "")),
                            "item": row.get("item", row.get("rec", [""])[0] if row.get("rec") else ""),
                            "response": row.get("response", row.get("pred", "")),
                            "ground_truth": row.get("resp", row.get("target", "")),
                        }, ensure_ascii=False) + "\n")
                print(f"[infer_emp] export -> {dst_json}")
            except Exception as exc:
                print(f"[infer_emp] export ECHEC: {exc}")
    else:
        print("run_infer_emp=False -> generation ECR[DialoGPT] ignoree.")
    return ok

## 6bis. Table 2 -- Llama 2-7B-Chat, ECR[Llama 2-Chat] LoRA et scorer local Qwen

L'article evalue 6 systemes (Table 2 + Appendix E) :

| Systeme | Dans ce notebook |
|---|---|
| `UniCRS` generation | fallback article (pas de ckpt vanille fourni) |
| `GPT-3.5-turbo-instruct` | fallback article (API payante) |
| `GPT-3.5-turbo` | fallback article (API payante) |
| `Llama 2-7B-Chat` zero-shot | **reproduit** (cellule ci-dessous) |
| `ECR[DialoGPT]` | **reproduit** (via `infer_emp.py`) |
| `ECR[Llama 2-Chat]` | **reproduit** via LoRA fine-tune (`src/ecr/llama_lora.py`) |

Le scorer GPT-4-turbo de l'Appendix E est remplace par **Qwen2.5-32B-Instruct-AWQ** servi via vLLM (gratuit, local, 4-bit AWQ ~20 GB sur 32 GB). On reporte un **Cohen kappa au niveau classement** entre ce scorer local et les annotateurs humains de l'article, pour documenter la validite du proxy.

In [ ]:
# Cellule 9b - Llama 2-7B-Chat zero-shot (baseline Table 2)
#
# Charge `test_data_dbpedia_emo.jsonl` (ou l'equivalent produit par prepare_redial),
# applique le template Llama 2 Chat (<s>[INST]<<SYS>>...[/INST]) et ecrit
# `results/generations/llama2_zero_shot.jsonl` (format attendu par le scorer).
#
# Deux backends :
#   - "hf" (defaut) : transformers.pipeline bf16 + Flash-Attn 2. Simple, fiable.
#     ~3-5 samples/s sur 5090 pour Llama-2-7B-Chat.
#   - "vllm" : serveur vLLM local. ~30-50 samples/s (10x). Besoin de ~15 GB VRAM
#     libre. Utile si > 500 samples. Le serveur est demarre par launch_llama_vllm.

_LLAMA_VLLM_PROC = [None]  # handle du process, pour teardown


def launch_llama_vllm(cfg):
    """Lance `vllm serve meta-llama/Llama-2-7b-chat-hf` en background.

    Retourne True si le serveur est pret, False sinon.
    Mutation du cfg : on NE change PAS `llama_backend` ici (responsabilite de
    l'appelant) pour garder la fonction pure.
    """
    if not cfg.get("llama_vllm_autostart"):
        return False
    if _LLAMA_VLLM_PROC[0] is not None and _LLAMA_VLLM_PROC[0].poll() is None:
        print("[llama-vllm] deja en cours")
        return True
    from src.ecr import scorer as sc
    port = cfg["llama_serve_port"]
    proc = sc.launch_vllm_server(
        model=cfg["llama_base_model"],
        port=port,
        quantization=None,  # Llama 2-7B-Chat en bf16 (pas de quant)
        max_model_len=2048,
        gpu_memory_utilization=0.45,
        log_path=cfg["logs_dir"] / "vllm_llama.log",
        extra_args=["--dtype", "bfloat16", "--enable-prefix-caching"],
    )
    if proc is None:
        print("[llama-vllm] echec demarrage -> fallback backend hf")
        return False
    _LLAMA_VLLM_PROC[0] = proc
    if not sc.wait_vllm_ready(port=port, timeout=300):
        print("[llama-vllm] timeout 5min -> fallback backend hf")
        proc.terminate()
        _LLAMA_VLLM_PROC[0] = None
        return False
    return True


def teardown_llama_vllm():
    """A appeler apres run_llama_lora_generate pour liberer la VRAM avant Qwen."""
    import atexit
    if _LLAMA_VLLM_PROC[0] is None:
        return
    proc = _LLAMA_VLLM_PROC[0]
    if proc.poll() is None:
        print("[llama-vllm] terminate")
        proc.terminate()
        try:
            proc.wait(timeout=30)
        except Exception:
            proc.kill()
    _LLAMA_VLLM_PROC[0] = None

def run_llama_zero_shot(cfg, limit=1000):
    """Genere les reponses Llama 2-7B-Chat zero-shot pour Table 2."""
    if cfg["dry_run"] or not cfg.get("run_llama_zero_shot"):
        print("run_llama_zero_shot desactive -> ligne Llama 2-7B-Chat en fallback article.")
        return None
    from src.ecr import llama_runner as lr

    gen_dir = Path(cfg["generations_dir"])
    gen_dir.mkdir(parents=True, exist_ok=True)
    out_path = gen_dir / "llama2_zero_shot.jsonl"
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[llama-zs] deja present: {out_path}")
        return out_path

    redial_gen = cfg["src_emo_dir"] / "data" / "redial_gen"
    candidates = [
        redial_gen / "test_data_dbpedia_emo.jsonl",
        redial_gen / "test_data_dbpedia.jsonl",
        redial_gen / "test_data.jsonl",
    ]
    jsonl = next((p for p in candidates if p.exists()), None)
    if jsonl is None:
        print(f"[llama-zs] test set introuvable parmi {[str(p) for p in candidates]}")
        return None

    samples = lr.load_test_samples(jsonl, limit=limit)
    print(f"[llama-zs] {len(samples)} exemples -> {out_path}")

    model_dir = cfg.get("llama_local_dir")
    model_dir = str(model_dir) if model_dir and Path(model_dir).exists() else cfg["llama_base_model"]

    if cfg.get("llama_backend") == "vllm":
        outputs = lr.generate_vllm(
            samples,
            base_url=f"http://localhost:{cfg['llama_serve_port']}/v1",
            model_name=cfg["llama_base_model"],
        )
    else:
        outputs = lr.generate_hf(
            samples,
            model_dir=model_dir,
            use_flash_attn_2=cfg.get("use_flash_attn_2", True),
        )

    # Enrichit chaque sortie avec l'history textuel (utile pour le scorer).
    for out, sample in zip(outputs, samples):
        out["history"] = sample.history
    lr.dump_generations(outputs, out_path)
    print(f"[llama-zs] OK -> {out_path}")

    # Liberation explicite de la VRAM pour la phase suivante (LoRA / scorer).
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
    except Exception:
        pass
    return out_path

In [ ]:
# Cellule 9c - ECR[Llama 2-Chat] : LoRA fine-tune + generation
#
# Corpus : `emo_data/llama_train.json` (2,459 reviews empathiques filtrees par
# les auteurs, Section 4.3 + 5.1) avec le prompt "emotion-aligned" (Eq. 8).
#
# Hyperparametres fideles a la Section 5.4 :
#   - LoRA r=16, alpha=32, dropout=0.05, modules q/k/v/o_proj ;
#   - 15 epochs, lr=5e-5, per_device_batch=4, grad_accum=4 -> eff batch 16 ;
#   - bf16 sur RTX 5090 (32 GB) ; 4-bit QLoRA dispo en fallback (cfg['lora_use_4bit']).
#
# Duree estimee : 2-3h sur 5090 bf16 (corpus tres petit).

def run_llama_lora_train(cfg):
    """Fine-tune le LoRA ECR[Llama 2-Chat] sur llama_train.json."""
    if cfg["dry_run"] or not cfg.get("run_llama_lora"):
        print("run_llama_lora desactive -> ligne ECR[Llama 2-Chat] en fallback article.")
        return None

    lora_dir = Path(cfg["llama_lora_dir"])
    adapter_file = lora_dir / "adapter_model.safetensors"
    if adapter_file.exists() or (lora_dir / "adapter_model.bin").exists():
        print(f"[lora] adapter deja entraine: {lora_dir}")
        return lora_dir

    train_json = cfg["emo_data_dir"] / "llama_train.json"
    if not train_json.exists():
        alt = cfg["src_emo_dir"] / "data" / "emo_data" / "llama_train.json"
        if alt.exists():
            train_json = alt
    if not train_json.exists():
        print(f"[lora] llama_train.json absent (cherche dans {train_json}) -> skip")
        return None

    from src.ecr.llama_lora import train_lora
    import os as _os

    hf_token = _os.environ.get("HF_TOKEN")
    base = cfg.get("llama_local_dir")
    base = str(base) if base and Path(base).exists() else cfg["llama_base_model"]

    print(f"[lora] base={base} train={train_json} -> {lora_dir}")
    result = train_lora(
        base_model=base,
        train_json=train_json,
        output_dir=lora_dir,
        lora_r=cfg["lora_r"],
        lora_alpha=cfg["lora_alpha"],
        lora_dropout=cfg["lora_dropout"],
        target_modules=cfg["lora_target_modules"],
        epochs=cfg["lora_epochs"],
        lr=cfg["lr_llama"],
        per_device_batch=cfg["lora_per_device_batch"],
        grad_accum=cfg["lora_grad_accum"],
        hf_token=hf_token,
        use_4bit=cfg.get("lora_use_4bit", False),
        use_flash_attn_2=cfg.get("use_flash_attn_2", True),
    )
    # Liberation VRAM : le modele base + LoRA reste dans `train_lora` closure.
    # On force un GC + empty_cache avant de recharger pour la generation.
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
    except Exception:
        pass
    return result


def run_llama_lora_generate(cfg, limit=1000):
    """Genere les reponses ECR[Llama 2-Chat] sur le test set."""
    if cfg["dry_run"] or not cfg.get("run_llama_lora"):
        return None

    gen_dir = Path(cfg["generations_dir"])
    gen_dir.mkdir(parents=True, exist_ok=True)
    out_path = gen_dir / "ecr_llama_chat.jsonl"
    if out_path.exists() and out_path.stat().st_size > 0:
        print(f"[lora-gen] deja present: {out_path}")
        return out_path

    lora_dir = Path(cfg["llama_lora_dir"])
    if not (lora_dir / "adapter_config.json").exists():
        print(f"[lora-gen] pas d'adapter dans {lora_dir} -> skip")
        return None

    redial_gen = cfg["src_emo_dir"] / "data" / "redial_gen"
    candidates = [
        redial_gen / "test_data_dbpedia_emo.jsonl",
        redial_gen / "test_data_dbpedia.jsonl",
    ]
    jsonl = next((p for p in candidates if p.exists()), None)
    if jsonl is None:
        print(f"[lora-gen] test set introuvable -> skip")
        return None

    from src.ecr import llama_runner as lr

    base = cfg.get("llama_local_dir")
    base = str(base) if base and Path(base).exists() else cfg["llama_base_model"]
    samples = lr.load_test_samples(jsonl, limit=limit)
    outputs = lr.generate_hf(
        samples,
        model_dir=base,
        lora_dir=str(lora_dir),
        use_flash_attn_2=cfg.get("use_flash_attn_2", True),
    )
    for out, sample in zip(outputs, samples):
        out["history"] = sample.history
    lr.dump_generations(outputs, out_path)
    print(f"[lora-gen] OK -> {out_path}")

    # Liberation VRAM avant la phase 4 (scorer Qwen 32B AWQ qui prend ~20 GB).
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
    except Exception:
        pass
    return out_path

In [ ]:
# Cellule 9d - Scorer Qwen2.5-32B-AWQ local (remplace GPT-4-turbo Appendix E)
#
# IMPORTANT : le serveur vLLM monopolise ~20 GB VRAM. Il DOIT etre lance APRES
# les phases d'entrainement et d'inference Llama (eviter tout conflit VRAM).
# On fait donc un `torch.cuda.empty_cache()` explicite avant, et on tue le
# process vLLM en fin de cellule.
#
# 4 systemes x 200 exemples x 5 rubriques = 4000 notes = ~2h15 sur 5090.

def run_llm_scorer(cfg):
    """Note chaque systeme avec Qwen2.5 et produit `subjective_metrics_llm.csv`."""
    if cfg["dry_run"] or not cfg.get("run_llm_scorer"):
        print("run_llm_scorer desactive -> subjective_metrics_llm.csv en fallback article.")
        return None

    gen_dir = Path(cfg["generations_dir"])
    systems = {
        "ECR[DialoGPT]":     gen_dir / "ecr_dialogpt.jsonl",
        "Llama 2-7B-Chat":   gen_dir / "llama2_zero_shot.jsonl",
        "ECR[Llama 2-Chat]": gen_dir / "ecr_llama_chat.jsonl",
    }
    available = {k: v for k, v in systems.items() if v.exists()}
    if not available:
        print("[scorer] aucune generation disponible -> fallback article")
        return None

    # Liberation agressive de la VRAM avant de lancer Qwen 32B AWQ (~20 GB).
    # On accepte de perdre le cache CUDA ; pire cas on le rebuild cote vLLM.
    try:
        import gc, torch as _torch
        gc.collect()
        if _torch.cuda.is_available():
            _torch.cuda.empty_cache()
            _torch.cuda.synchronize()
            free, total = _torch.cuda.mem_get_info()
            free_gb = free / 1e9
            total_gb = total / 1e9
            print(f"[scorer] VRAM libre avant vLLM : {free_gb:.1f}/{total_gb:.1f} GB")
            if free_gb < 22:
                print("[scorer] WARN VRAM < 22 GB, le serveur AWQ risque un OOM")
    except Exception as exc:
        print(f"[scorer] check VRAM echoue (non bloquant): {exc}")

    from src.ecr import scorer as sc
    import json as _json
    import time as _time
    import atexit

    proc = sc.launch_vllm_server(
        model=cfg["scorer_model"],
        port=cfg["scorer_port"],
        quantization="awq",
        max_model_len=cfg["scorer_max_model_len"],
        gpu_memory_utilization=0.9,
        log_path=cfg["logs_dir"] / "vllm_scorer.log",
    )
    if proc is None:
        print("[scorer] serveur vLLM n'a pas pu demarrer -> fallback article")
        return None
    atexit.register(lambda: proc.terminate() if proc.poll() is None else None)
    try:
        if not sc.wait_vllm_ready(port=cfg["scorer_port"], timeout=900):
            print("[scorer] serveur vLLM non pret apres 15min -> abandon")
            return None

        from openai import OpenAI
        client = OpenAI(base_url=f"http://localhost:{cfg['scorer_port']}/v1", api_key="EMPTY")

        per_system = {}
        for sys_name, path in available.items():
            rows = []
            with path.open() as f:
                for line in f:
                    rows.append(_json.loads(line))
            print(f"[scorer] {sys_name}: {len(rows)} exemples dispo, sample={cfg['scorer_sample_size']}")
            t0 = _time.time()
            df = sc.score_system(
                client,
                model=cfg["scorer_model"],
                generations=rows,
                sample_size=cfg["scorer_sample_size"],
                seed=cfg["seed"],
                max_workers=cfg.get("scorer_concurrency", 16),
            )
            dt = _time.time() - t0
            print(f"[scorer] {sys_name}: {len(df)} notes en {dt/60:.1f}min")
            per_system[sys_name] = df
            df.to_csv(cfg["results_dir"] / f"scorer_raw_{sys_name.replace(' ', '_').replace('[','').replace(']','')}.csv", index=False)

        df_subj = sc.aggregate_subjective(per_system)
        df_subj.to_csv(cfg["results_dir"] / cfg["subjective_llm_file"], index=False)
        print(f"[scorer] -> {cfg['results_dir'] / cfg['subjective_llm_file']}")
        print(df_subj)
        return df_subj
    finally:
        if proc.poll() is None:
            proc.terminate()
            try:
                proc.wait(timeout=30)
            except Exception:
                proc.kill()

In [ ]:
# Cellule 9e - Cohen kappa scorer local vs annotateurs humains (Section 5.6)
#
# On compare le CLASSEMENT (rank) des systemes par metrique, pas les valeurs
# absolues (echelles differentes : LLM 0-9 strict, humains 0-9 mais biais
# differents). Kappa > 0.6 = accord substantiel (chiffre reference article).

def compute_scorer_kappa(cfg, df_subj_llm, df_subj_human):
    if cfg["dry_run"] or not cfg.get("run_kappa"):
        return None
    try:
        from sklearn.metrics import cohen_kappa_score
    except ImportError:
        print("[kappa] scikit-learn indisponible -> skip")
        return None

    metrics = ["Emo Int", "Emo Pers", "Log Pers", "Info", "Life"]
    rows = []
    for m in metrics:
        if m not in df_subj_llm.columns or m not in df_subj_human.columns:
            continue
        llm = df_subj_llm.set_index("Model")[m]
        human = df_subj_human.set_index("Model")[m]
        common = llm.index.intersection(human.index)
        if len(common) < 3:
            continue
        rk_llm = llm[common].rank(method="min").astype(int)
        rk_human = human[common].rank(method="min").astype(int)
        kappa = cohen_kappa_score(rk_llm.tolist(), rk_human.tolist())
        rows.append({"metric": m, "n_models": len(common), "kappa_rank": kappa})
    df_kappa = pd.DataFrame(rows)
    if not df_kappa.empty:
        df_kappa.to_csv(cfg["results_dir"] / "scorer_kappa.csv", index=False)
        print("=== Cohen kappa scorer local (Qwen2.5-32B-AWQ) vs humains article ===")
        print(df_kappa)
        mean_k = df_kappa["kappa_rank"].mean()
        level = "SUBSTANTIEL" if mean_k > 0.6 else ("MODERE" if mean_k > 0.4 else "FAIBLE")
        print(f"kappa moyen = {mean_k:.3f} ({level})")
    return df_kappa

In [ ]:
# Cellule 9f - Parse des logs train_rec -> objective_metrics.csv + ablation_metrics.csv
#
# A lancer APRES run_recommendation_sweep. Extrait les 7 metriques Table 1 de
# chaque log et complete avec les valeurs article pour les baselines non
# reproduites (KBRD, KGSF, RevCore, UCCR).
#
# Cette fonction est robuste a l'absence de logs : si aucun run du sweep n'a
# produit de log parsable, elle retourne None et load_results_data utilisera
# le fallback article. AUCUNE exception n'est propagee pour ne pas casser le
# pipeline (on accepte une Table 1 incomplete).

def build_results_from_logs(cfg):
    if cfg["dry_run"]:
        return None
    try:
        from src.ecr.metrics_parser import write_rec_results
    except ImportError as exc:
        print(f"[parse] import metrics_parser echoue: {exc}")
        return None

    logs_dir = Path(cfg["logs_dir"])
    if not logs_dir.exists():
        print(f"[parse] {logs_dir} absent -> aucun log a parser")
        return None

    run_names = [v["name"] for v in REC_VARIANTS]
    available_logs = [n for n in run_names if (logs_dir / f"train_rec_{n}.log").exists()]
    if not available_logs:
        print(f"[parse] aucun log train_rec_*.log dans {logs_dir} -> fallback article")
        return None

    # _fallback_objective est defini dans code-17-load (cellule suivante) ;
    # on gere le cas ou il n'est pas encore charge (execution hors-ordre).
    try:
        fallback = _fallback_objective()
    except NameError:
        fallback = None

    try:
        df_obj = write_rec_results(
            logs_dir=logs_dir,
            results_dir=cfg["results_dir"],
            run_names=run_names,
            fallback_objective=fallback,
        )
    except Exception as exc:
        print(f"[parse] echec write_rec_results: {exc}")
        return None

    if df_obj is not None and not df_obj.empty:
        print("=== Table 1 (objective_metrics.csv) ===")
        print(df_obj)
    return df_obj

## 7. Chargement des resultats (Tables 1-3)

`load_results_data` lit des CSV facultatifs dans `results/`. En l'absence de fichiers (cas le plus courant : `DRY_RUN=True`), on retombe sur les valeurs publiees par l'article afin que le notebook reste reproductible :

- Table 1 — recommandation objective (AUC, RT@n, R@n) ;
- Table 2 — generation subjective (Emo Int, Emo Pers, Log Pers, Info, Life) ;
- Table 3 — etude d'ablation (UniCRS, ECR[L], ECR[LS], ECR[LG], ECR).

In [ ]:
# Cellule 10 - Chargement des metriques (repli sur les tables de l'article)
def _fallback_objective():
    return pd.DataFrame([
        {"Model": "KBRD",    "AUC": 0.503, "RT@1": 0.040, "RT@10": 0.182, "RT@50": 0.381, "R@1": 0.037, "R@10": 0.175, "R@50": 0.360},
        {"Model": "KGSF",    "AUC": 0.513, "RT@1": 0.043, "RT@10": 0.195, "RT@50": 0.383, "R@1": 0.040, "R@10": 0.182, "R@50": 0.361},
        {"Model": "RevCore", "AUC": 0.514, "RT@1": 0.054, "RT@10": 0.230, "RT@50": 0.410, "R@1": 0.046, "R@10": 0.209, "R@50": 0.390},
        {"Model": "UCCR",    "AUC": 0.499, "RT@1": 0.038, "RT@10": 0.208, "RT@50": 0.423, "R@1": 0.039, "R@10": 0.198, "R@50": 0.407},
        {"Model": "UniCRS",  "AUC": 0.506, "RT@1": 0.052, "RT@10": 0.229, "RT@50": 0.439, "R@1": 0.047, "R@10": 0.212, "R@50": 0.414},
        {"Model": "ECR",     "AUC": 0.541, "RT@1": 0.055, "RT@10": 0.238, "RT@50": 0.452, "R@1": 0.049, "R@10": 0.220, "R@50": 0.428},
    ])


def _fallback_subjective_llm():
    return pd.DataFrame([
        {"Model": "UniCRS",                 "Emo Int": 0.400, "Emo Pers": 0.942, "Log Pers": 0.793, "Info": 0.673, "Life": 2.241},
        {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 1.706, "Emo Pers": 3.043, "Log Pers": 3.474, "Info": 2.975, "Life": 4.182},
        {"Model": "GPT-3.5-turbo",          "Emo Int": 2.215, "Emo Pers": 3.754, "Log Pers": 4.782, "Info": 4.147, "Life": 5.338},
        {"Model": "Llama 2-7B-Chat",        "Emo Int": 3.934, "Emo Pers": 6.030, "Log Pers": 5.886, "Info": 5.904, "Life": 7.129},
        {"Model": "ECR[DialoGPT]",          "Emo Int": 4.011, "Emo Pers": 4.878, "Log Pers": 4.736, "Info": 5.094, "Life": 5.906},
        {"Model": "ECR[Llama 2-Chat]",      "Emo Int": 6.826, "Emo Pers": 7.724, "Log Pers": 6.702, "Info": 7.653, "Life": 8.063},
    ])


def _fallback_subjective_human():
    return pd.DataFrame([
        {"Model": "UniCRS",                 "Emo Int": 0.947, "Emo Pers": 0.775, "Log Pers": 1.158, "Info": 0.380, "Life": 1.805},
        {"Model": "GPT-3.5-turbo-instruct", "Emo Int": 2.048, "Emo Pers": 2.555, "Log Pers": 3.265, "Info": 1.822, "Life": 3.648},
        {"Model": "GPT-3.5-turbo",          "Emo Int": 2.890, "Emo Pers": 3.678, "Log Pers": 5.323, "Info": 3.233, "Life": 5.125},
        {"Model": "Llama 2-7B-Chat",        "Emo Int": 4.432, "Emo Pers": 6.152, "Log Pers": 6.393, "Info": 5.713, "Life": 7.463},
        {"Model": "ECR[DialoGPT]",          "Emo Int": 5.097, "Emo Pers": 4.817, "Log Pers": 5.398, "Info": 4.628, "Life": 6.385},
        {"Model": "ECR[Llama 2-Chat]",      "Emo Int": 7.130, "Emo Pers": 7.575, "Log Pers": 7.403, "Info": 7.172, "Life": 8.468},
    ])


def _fallback_ablation():
    return pd.DataFrame([
        {"Model": "UniCRS",  "AUC": 0.506, "RT@10": 0.229, "RT@50": 0.439, "R@10": 0.212, "R@50": 0.414},
        {"Model": "ECR[L]",  "AUC": 0.535, "RT@10": 0.229, "RT@50": 0.444, "R@10": 0.213, "R@50": 0.420},
        {"Model": "ECR[LS]", "AUC": 0.541, "RT@10": 0.232, "RT@50": 0.442, "R@10": 0.216, "R@50": 0.420},
        {"Model": "ECR[LG]", "AUC": 0.535, "RT@10": 0.232, "RT@50": 0.453, "R@10": 0.216, "R@50": 0.428},
        {"Model": "ECR",     "AUC": 0.541, "RT@10": 0.238, "RT@50": 0.452, "R@10": 0.220, "R@50": 0.428},
    ])


def load_results_data(cfg):
    """Charge les metriques depuis `results/` ou retombe sur les tables de l'article."""
    r = cfg["results_dir"]
    r.mkdir(parents=True, exist_ok=True)
    pairs = [
        (r / cfg["objective_file"], _fallback_objective),
        (r / cfg["subjective_llm_file"], _fallback_subjective_llm),
        (r / cfg["subjective_human_file"], _fallback_subjective_human),
        (r / cfg["ablation_file"], _fallback_ablation),
    ]
    frames = []
    for path, fallback in pairs:
        if path.exists():
            print(f"[load] {path.name} depuis results/")
            frames.append(pd.read_csv(path))
        else:
            print(f"[load] {path.name} absent -> fallback article")
            frames.append(fallback())
    return tuple(frames)

In [ ]:
# Cellule 11 - EDA : distribution de feedback, labels d'emotion, couverture de reviews
def build_dataset_eda(cfg):
    """Produit les dataframes d'EDA alignes avec la Section 5.1 de l'article."""
    df_feedback = pd.DataFrame({
        "feedback": ["like", "dislike", "not say"],
        "count":    [8110,   490,      1400],  # proportions 81.1 / 4.9 / 14.0 %
    })
    # Estimation des parts des 9 labels d'emotion (Section 4.1.1 - negative = 8%)
    df_emotions = pd.DataFrame({
        "emotion": ["like", "curious", "happy", "grateful", "negative",
                      "neutral", "nostalgia", "agreement", "surprise"],
        "share":   [28.0,   14.0,      12.0,    10.0,       8.0,
                     11.0,    5.0,       7.0,        5.0],  # approximation pedagogique
    })
    df_reviews = pd.DataFrame([
        {"backbone": "DialoGPT",         "reviews": 34953, "movies": 4092},
        {"backbone": "Llama 2-7B-Chat",  "reviews": 2459,  "movies": 1553},
    ])
    df_weights = pd.DataFrame([
        {"feedback": k, "weight": v} for k, v in cfg["feedback_weights"].items()
    ])
    return df_feedback, df_emotions, df_reviews, df_weights


def describe_eda(df_feedback, df_emotions, df_reviews):
    print("=== Feedback distribution (Table Section 5.1) ===")
    print(df_feedback)
    print("\n=== Emotion labels (Section 4.1.1) ===")
    print(df_emotions)
    print("\n=== IMDb review coverage (Section 5.1) ===")
    print(df_reviews)

In [ ]:
# Cellule 12 - Analyse et table comparative
def evaluate_results(df_obj, df_subj_llm, df_subj_human):
    best_auc = df_obj.loc[df_obj["AUC"].idxmax()]
    best_rt10 = df_obj.loc[df_obj["RT@10"].idxmax()]
    best_life_llm = df_subj_llm.loc[df_subj_llm["Life"].idxmax()]
    best_life_human = df_subj_human.loc[df_subj_human["Life"].idxmax()]
    return pd.DataFrame([
        {"metric": "best_auc",        "model": best_auc["Model"],        "value": best_auc["AUC"]},
        {"metric": "best_rt10",       "model": best_rt10["Model"],       "value": best_rt10["RT@10"]},
        {"metric": "best_life_llm",   "model": best_life_llm["Model"],   "value": best_life_llm["Life"]},
        {"metric": "best_life_human", "model": best_life_human["Model"], "value": best_life_human["Life"]},
    ])


def build_comparison_table(df_obj, df_subj_llm, df_subj_human):
    llm = df_subj_llm.add_suffix("_llm").rename(columns={"Model_llm": "Model"})
    human = df_subj_human.add_suffix("_human").rename(columns={"Model_human": "Model"})
    return df_obj.merge(llm, on="Model", how="left").merge(human, on="Model", how="left")


def load_hyperparam_sweep(cfg):
    """Charge `results/sweep_like_weight.csv` si present, sinon DataFrame vide.

    Appendix B non reproduit faute de budget GPU : un vrai sweep sur
    `like_weight in {1.0, 1.5, 2.0, 2.5, 3.0}` demanderait 5 trainings
    supplementaires (~10h GPU de plus). La fonction de simulation precedente
    fabriquait des chiffres autour de la baseline ECR : on la retire par
    honnetete scientifique. Si tu veux reproduire, lance 5 fois
    `run_recommendation_sweep` en variant `cfg['feedback_weights']['like']`
    et ecris les AUC dans `results/sweep_like_weight.csv` (colonnes :
    like_weight, AUC).
    """
    path = cfg["results_dir"] / "sweep_like_weight.csv"
    if path.exists():
        return pd.read_csv(path)
    return pd.DataFrame(columns=["like_weight", "AUC"])


# Retro-compat : pointeur vers la nouvelle fonction pour ne pas casser le
# pipeline principal s'il reference encore `simulate_hyperparam_sweep`.
def simulate_hyperparam_sweep(df_obj):
    """Deprecie : utiliser `load_hyperparam_sweep(cfg)` a la place."""
    print("[deprecate] simulate_hyperparam_sweep retire (valeurs fabriquees). "
          "Utilise load_hyperparam_sweep(cfg) avec un CSV reel.")
    return pd.DataFrame(columns=["like_weight", "AUC"])

## 8. Pipeline principal (**une seule cellule d'import de donnees**)

Cette cellule enchaine la preparation legere (clone + archives + donnees ReDial) et le chargement des metriques. Avec `DRY_RUN=True`, aucun script d'entrainement n'est lance mais on obtient des DataFrames prets pour la visualisation.

In [ ]:
# Cellule 13 - IMPORT DES DONNEES (pipeline principal, cellule unique)
cfg = get_config()

# 1. Environnement ECR
clone_ecr_repo(cfg)
patch_ecr_compat(cfg)       # compat transformers >= 4.40 / pyg >= 2.5 + patches 1-21
download_external_assets(cfg)

# 2. Modeles de base HF (DialoGPT-small + RoBERTa) dans src_emo/save/
#    + installation optionnelle des poids pre-entraines (flag use_pretrained_ckpt)
if not cfg["dry_run"]:
    install_base_models(cfg)
    install_pretrained_ckpts(cfg)

# 3. Preparation des donnees ReDial (Section 4.1)
prepare_redial_data(cfg)

# 4a. Emotional Semantic Fusion (train_pre.py) - skip si use_pretrained_ckpt=True
run_emotional_semantic_fusion(cfg)

# 4b. Phase 2 - Table 1 + Table 3 : sweep sur 5 variantes (UniCRS, ECR[L], ECR[LS], ECR[LG], ECR)
#     ~10h GPU sur 5090. Chaque run produit logs/train_rec_<name>.log.
if cfg.get("run_rec_sweep"):
    run_recommendation_sweep(cfg)
else:
    run_recommendation_training(cfg)  # variante unique ECR complete (retro-compat)

# 4c. Parse les logs des runs -> results/objective_metrics.csv + ablation_metrics.csv
build_results_from_logs(cfg)

# 4d. ECR[DialoGPT] : train_emp + infer_emp + export generations/ecr_dialogpt.jsonl
run_response_generation_training(cfg)

# 4e. Phase 3 - Table 2 : Llama 2-7B-Chat zero-shot + ECR[Llama 2-Chat] LoRA
#     Optionnel : demarre un serveur vLLM Llama pour 10x de throughput sur la
#     generation (via cfg["llama_vllm_autostart"]=True). Sinon backend HF bf16
#     avec Flash-Attention 2 (si dispo).
if cfg.get("llama_vllm_autostart"):
    if launch_llama_vllm(cfg):
        cfg["llama_backend"] = "vllm"
run_llama_zero_shot(cfg)
run_llama_lora_train(cfg)
run_llama_lora_generate(cfg)
# Liberer la VRAM vLLM Llama avant Qwen scorer
if cfg.get("llama_vllm_autostart"):
    teardown_llama_vllm()

# 4f. Phase 4 - scorer Qwen2.5-32B-AWQ (remplace GPT-4-turbo Appendix E)
#     ATTENTION : le serveur vLLM monopolise la VRAM, il est lance APRES
#     toutes les phases d'entrainement/inference LLM.
#     Double barriere GC pour s'assurer que plus rien ne detient de CUDA mem.
import gc as _gc
try:
    import torch as _torch
    _gc.collect()
    if _torch.cuda.is_available():
        _torch.cuda.empty_cache()
        _torch.cuda.synchronize()
except Exception:
    pass
run_llm_scorer(cfg)

# 5. Chargement des metriques (CSV reels ou repli article)
df_obj, df_subj_llm, df_subj_human, df_ablation = load_results_data(cfg)
df_feedback, df_emotions, df_reviews, df_weights = build_dataset_eda(cfg)

# 6. Cohen kappa scorer local vs humains (Section 5.6)
df_kappa = compute_scorer_kappa(cfg, df_subj_llm, df_subj_human)

describe_eda(df_feedback, df_emotions, df_reviews)
df_obj.head()

## 8bis. Limitations et ecarts methodologiques

Certaines lignes des tables de l'article n'ont pas pu etre reproduites dans ce notebook faute d'API payantes et de personnel d'annotation. Elles sont conservees en **valeur article** (fallback) dans les visualisations pour permettre la comparaison. Les substitutions documentees sont :

| Article | Ce notebook | Justification |
|---|---|---|
| `GPT-3.5-turbo-instruct` / `GPT-3.5-turbo` (Tables 1, 2) | Non execute -> fallback article | Acces OpenAI payant, hors perimetre projet de session |
| Scorer `GPT-4-turbo` (Section 5.6 + Appendix E) | `Qwen/Qwen2.5-32B-Instruct-AWQ` servi via vLLM (local, 4-bit) | Scorer open-source ; proxy valide via Cohen kappa au niveau classement (cf. `results/scorer_kappa.csv`) |
| Baselines `KBRD` / `KGSF` / `RevCore` / `UCCR` (Table 1) | Non reentrainees -> fallback article | Codebases PyTorch 1.x / Python 3.6 ; port non realisable dans le temps imparti |
| 3 annotateurs humains sur 200 exemples (Table 2) | Non realise | Hors perimetre projet de session |
| Cohen kappa inter-annotateurs humain-humain (Section 5.6) | Remplace par Cohen kappa scorer-vs-humains article | Proxy indirect de la qualite du scorer local |
| Appendix B sweep `like_weight` | Non reproduit (`simulate_hyperparam_sweep` retire) | ~10h GPU de plus ; CSV `results/sweep_like_weight.csv` a produire manuellement |

**Reproduit reellement :**

- Table 1 lignes `UniCRS` et `ECR` (2/6 mesurees) ;
- Table 3 complete (5/5 variantes d'ablation mesurees) ;
- Table 2 lignes `ECR[DialoGPT]`, `Llama 2-7B-Chat`, `ECR[Llama 2-Chat]` (3/6 mesurees) ;
- Scoring local Qwen2.5-32B-AWQ sur 200 exemples x 5 rubriques (Appendix E) ;
- Cohen kappa classement scorer vs humains article.

Les plots `plot_*` qui melent lignes mesurees et lignes fallback sont clairement etiquetes dans leur legende (`Model` tel quel, y compris pour les lignes non reproduites). Les artefacts mesures sont dans `results/` (CSV + `generations/` + `logs/`), les valeurs fallback sont dans le code (`_fallback_*`) pour garder le notebook executable sans reseau.

## 9. Compilation finale et visualisations comparatives

On recapitule les resultats (meilleurs scores, table comparative) puis on affiche tous les graphiques definis dans `src/viz/plots.py`. Les visualisations suivent les figures et tableaux de l'article :

- `plot_feedback_distribution`, `plot_feedback_weights`, `plot_emotion_label_distribution`, `plot_review_coverage` → Section 4.1 & 5.1 ;
- `plot_objective_metrics`, `plot_model_rankings`, `plot_ablation_study` → Tables 1 et 3 ;
- `plot_subjective_metrics`, `plot_subjective_radar`, `plot_llm_vs_human_correlation` → Table 2 & Section 5.6 ;
- `plot_training_loss`, `plot_hyperparam_sweep` → diagnostics d'entrainement et Appendix B.

In [ ]:
# Cellule 14 - Compilation finale : resume + visualisations
summary_df = evaluate_results(df_obj, df_subj_llm, df_subj_human)
comparison_df = build_comparison_table(df_obj, df_subj_llm, df_subj_human)

print("=== Summary ===")
print(summary_df)
print("\n=== Comparison table ===")
print(comparison_df.head())

# Dataset / EDA (Sections 4.1 et 5.1)
plot_feedback_distribution(df_feedback)
plot_feedback_weights(cfg["feedback_weights"])
plot_emotion_label_distribution(df_emotions)
plot_review_coverage(df_reviews)

# Recommandation (Table 1 + ablation Table 3)
plot_model_rankings(df_obj, metric="AUC")
plot_objective_metrics(df_obj)
plot_ablation_study(df_ablation)

# Generation de reponses (Table 2 + LLM vs Human)
plot_subjective_metrics(df_subj_llm, "Subjective metrics - LLM scorer (Qwen2.5-32B-Instruct-AWQ, local)")
plot_subjective_metrics(df_subj_human, "Subjective metrics - Human annotators (article, fallback)")
plot_subjective_radar(df_subj_llm, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (Qwen scorer)")
plot_subjective_radar(df_subj_human, "ECR[Llama 2-Chat]", "Radar - ECR[Llama 2-Chat] (human, article)")
plot_llm_vs_human_correlation(df_subj_llm, df_subj_human)

# Cohen kappa classement scorer local vs humains article
if df_kappa is not None and not df_kappa.empty:
    print("\n=== Cohen kappa rank-level (scorer Qwen vs humains article) ===")
    print(df_kappa)
    print(f"kappa moyen = {df_kappa['kappa_rank'].mean():.3f}")

# Diagnostics d'entrainement (si l'utilisateur a entraine les modeles)
history_path = cfg["results_dir"] / "training_history.csv"
if history_path.exists():
    df_history = pd.read_csv(history_path)
    plot_training_loss(df_history)
else:
    print(f"[info] {history_path.name} absent -> `plot_training_loss` ignore (disponible apres un vrai entrainement).")

# Sensibilite au poids `like` (Appendix B) : reel si `results/sweep_like_weight.csv` present, sinon skip.
df_sweep = load_hyperparam_sweep(cfg)
if not df_sweep.empty:
    plot_hyperparam_sweep(df_sweep, param="like_weight", metric="AUC")
else:
    print("[info] sweep_like_weight.csv absent -> Appendix B non reproduit (cf. cellule Limitations).")